# idd-travel-time · Global Cities & Informal Transit

This notebook demonstrates the library's primary design goal: computing meaningful
travel-time isochrones in cities where **no GTFS feed exists** and transit
operates informally.

**Cities covered**

| City | Country | Transit type | Preset |
|---|---|---|---|
| Lagos | Nigeria | Danfo minibus | `informal_minibus` |
| Nairobi | Kenya | Matatu minibus | `informal_minibus` |
| Abuja | Nigeria | BRT + keke napep | `mototaxi` |
| Kampala | Uganda | Boda-boda motorcycle taxi | `mototaxi` |
| Jakarta | Indonesia | Bajaj / ojek | `tuk_tuk` |
| Dhaka | Bangladesh | CNG auto-rickshaw | `tuk_tuk` |

> **Design note:** idd-travel-time uses *typical-speed assumptions* rather than
> schedule-exact routing. This is intentional — it supports global coverage
> including cities where GTFS data does not exist. See `TransitProfile` and
> `profiles.PRESETS` for the full parameter set.


## Setup

In [ ]:
from idd_travel_time.constants import TransportMode, TimeContext, VizLayer
from idd_travel_time.models import TravelTimeRequest, TransitProfile
from idd_travel_time.isochrone import compute
from idd_travel_time import viz
from idd_travel_time.profiles import list_presets, get_preset

# Print all available presets and their speed ranges
print("Available TransitProfile presets:\n")
for name in list_presets():
    p = get_preset(name)
    lo, hi = p.typical_speed_kph
    print(f"  {name:<22} {lo:4.0f}–{hi:4.0f} kph  [{p.coverage.value}]")


## 1 · Lagos — informal minibus (danfo)

Lagos has one of the world's largest informal transit networks. Danfo minibuses
operate on fixed routes but with no published timetable. We model them with the
`informal_minibus` preset, adjusted for rush-hour congestion.


In [ ]:
# Lagos Island — Tafawa Balewa Square
lagos_profile = TransitProfile.from_preset(
    "informal_minibus",
    time_context=TimeContext.RUSH_HOUR,
    typical_speed_kph=(10.0, 18.0),   # slower in rush-hour Lagos traffic
)

lagos_request = TravelTimeRequest(
    origin_lat=6.4541,
    origin_lon=3.3947,
    mode=TransportMode.TRANSIT,
    duration_minutes=30,
    time_context=TimeContext.RUSH_HOUR,
    transit_profile=lagos_profile,
)

print(f"Transit profile: {lagos_profile.mode_name}")
print(f"Speed range: {lagos_profile.typical_speed_kph} kph")
print(f"Midpoint: {lagos_profile.midpoint_speed_kph:.1f} kph")


In [ ]:
lagos_result = compute(lagos_request)
viz.render(lagos_result, layers=[VizLayer.ISOCHRONE, VizLayer.HEATMAP])


## 2 · Nairobi — matatu (off-peak vs rush hour)

We compare two profiles for the same origin to show how time-of-day context
affects the reachable area. Matatus are faster off-peak when traffic is light.


In [ ]:
origin = dict(origin_lat=-1.2864, origin_lon=36.8172)

# Off-peak: matatus move freely
off_peak_profile = TransitProfile.from_preset(
    "informal_minibus",
    time_context=TimeContext.OFF_PEAK,
)

# Rush hour: significant congestion on Uhuru Highway / Mombasa Road
rush_profile = TransitProfile.from_preset(
    "informal_minibus",
    time_context=TimeContext.RUSH_HOUR,
    typical_speed_kph=(8.0, 15.0),
)

req_off = TravelTimeRequest(
    **origin, mode=TransportMode.TRANSIT, duration_minutes=30,
    time_context=TimeContext.OFF_PEAK, transit_profile=off_peak_profile,
)
req_rush = TravelTimeRequest(
    **origin, mode=TransportMode.TRANSIT, duration_minutes=30,
    time_context=TimeContext.RUSH_HOUR, transit_profile=rush_profile,
)

result_off = compute(req_off)
result_rush = compute(req_rush)

import geopandas as gpd

def area_km2(result):
    proj = result.isochrone_gdf.to_crs(result.isochrone_gdf.estimate_utm_crs())
    return proj.geometry.area.iloc[0] / 1e6

print(f"Off-peak reachable area:  {area_km2(result_off):.2f} km²")
print(f"Rush-hour reachable area: {area_km2(result_rush):.2f} km²")
print(f"Reduction:                {100 * (1 - area_km2(result_rush)/area_km2(result_off)):.0f}%")


In [ ]:
# Visualise the two isochrones overlaid using folium directly
import folium

m = folium.Map(location=[-1.2864, 36.8172], zoom_start=12, tiles="CartoDB positron")

folium.GeoJson(
    result_off.isochrone_gdf.__geo_interface__,
    name="Off-peak (30 min)",
    style_function=lambda _: {"color": "#2ca02c", "fillColor": "#2ca02c",
                               "fillOpacity": 0.2, "weight": 2},
).add_to(m)

folium.GeoJson(
    result_rush.isochrone_gdf.__geo_interface__,
    name="Rush hour (30 min)",
    style_function=lambda _: {"color": "#d62728", "fillColor": "#d62728",
                               "fillOpacity": 0.2, "weight": 2},
).add_to(m)

folium.Marker([-1.2864, 36.8172], tooltip="Origin", icon=folium.Icon(color="black")).add_to(m)
folium.LayerControl().add_to(m)
m


## 3 · Kampala — boda-boda motorcycle taxi

Boda-bodas are the dominant transit mode in Kampala. They use the full road
network and can filter through traffic — modelled with the `mototaxi` preset.


In [ ]:
# Kampala City Square
kampala_profile = TransitProfile.from_preset("mototaxi")

kampala_request = TravelTimeRequest(
    origin_lat=0.3163,
    origin_lon=32.5822,
    mode=TransportMode.TRANSIT,
    duration_minutes=20,
    time_context=TimeContext.OFF_PEAK,
    transit_profile=kampala_profile,
)

kampala_result = compute(kampala_request)
viz.render(kampala_result, layers=[VizLayer.ISOCHRONE, VizLayer.NETWORK])


## 4 · Jakarta — bajaj / ojek (tuk-tuk)

Jakarta's bajaj three-wheelers and ojek motorcycle taxis use the road network
freely. We use the `tuk_tuk` preset — similar speed to an informal minibus but
with better penetration into smaller streets.


In [ ]:
# Monas (National Monument), Central Jakarta
jakarta_profile = TransitProfile.from_preset("tuk_tuk")

jakarta_request = TravelTimeRequest(
    origin_lat=-6.1754,
    origin_lon=106.8272,
    mode=TransportMode.TRANSIT,
    duration_minutes=25,
    time_context=TimeContext.OFF_PEAK,
    transit_profile=jakarta_profile,
)

jakarta_result = compute(jakarta_request)
viz.render(jakarta_result)


## 5 · Custom TransitProfile — fully manual

All preset parameters are overridable. You can also construct a `TransitProfile`
from scratch for a city or context with no matching preset.


In [ ]:
# Hypothetical electric tuk-tuk pilot in Addis Ababa:
# faster than a bajaj, uses fixed feeder routes from the new LRT stations
custom_profile = TransitProfile(
    typical_speed_kph=(20.0, 30.0),
    coverage=TransitProfile.__dataclass_fields__  # road_network
)


In [ ]:
from idd_travel_time.constants import TransitCoverage

addis_profile = TransitProfile(
    typical_speed_kph=(20.0, 30.0),
    coverage=TransitCoverage.ROAD_NETWORK,
    time_context=TimeContext.OFF_PEAK,
    mode_name="Electric tuk-tuk pilot (Addis Ababa)",
)

addis_request = TravelTimeRequest(
    origin_lat=9.0308,
    origin_lon=38.7497,   # Meskel Square
    mode=TransportMode.TRANSIT,
    duration_minutes=20,
    time_context=TimeContext.OFF_PEAK,
    transit_profile=addis_profile,
)

addis_result = compute(addis_request)
print(f"Profile: {addis_profile.mode_name}")
print(f"Speed midpoint: {addis_profile.midpoint_speed_kph:.1f} kph")

viz.render(addis_result, layers=[VizLayer.ISOCHRONE, VizLayer.HEATMAP])


## 6 · Batch computation across multiple cities

`compute_batch()` accepts a list of requests and reuses a single router instance,
which avoids repeated initialisation overhead on cluster runs.


In [ ]:
from idd_travel_time.isochrone import compute_batch
from idd_travel_time.routers import OsmnxRouter

cities = [
    ("Lagos",    6.4541,  3.3947,  "informal_minibus"),
    ("Nairobi",  -1.2864, 36.8172, "informal_minibus"),
    ("Kampala",  0.3163,  32.5822, "mototaxi"),
    ("Abuja",    9.0579,  7.4951,  "mototaxi"),
]

requests = [
    TravelTimeRequest(
        origin_lat=lat, origin_lon=lon,
        mode=TransportMode.TRANSIT,
        duration_minutes=30,
        transit_profile=TransitProfile.from_preset(preset),
    )
    for name, lat, lon, preset in cities
]

router = OsmnxRouter()
results = compute_batch(requests, router=router)

for (name, *_), result in zip(cities, results):
    proj = result.isochrone_gdf.to_crs(result.isochrone_gdf.estimate_utm_crs())
    a = proj.geometry.area.iloc[0] / 1e6
    print(f"{name:<10}  {a:6.2f} km²  reachable in 30 min by transit")


## 7 · Saving all maps

Save each city's map to an HTML file for sharing or embedding in reports.


In [ ]:
from pathlib import Path

output_dir = Path("/tmp/idd-travel-time-cities")
output_dir.mkdir(exist_ok=True)

for (name, *_), result in zip(cities, results):
    out = output_dir / f"{name.lower()}_transit_30min.html"
    saved = viz.save(result, out, layers=[VizLayer.ISOCHRONE, VizLayer.HEATMAP])
    print(f"Saved: {saved}")
